# `image_split` 노트북 독립 실행

- **의존성**: `opencv-python`, `numpy`, `boto3`(S3 사용 시)
- 노트북 **작업 디렉터리**를 저장소 루트(`product_semantic_search`)로 맞추세요.
- 코드는 `notebooks/image_split_standalone.py`에 있으며, 아래 셀에서 **모듈로 로드**합니다.

In [ ]:
# 필요 시 한 번만
# %pip install opencv-python numpy boto3

In [ ]:
import importlib.util
from pathlib import Path

# 저장소 루트가 cwd가 아니면 아래 주석 해제 후 경로 수정
# import os
# os.chdir(r"c:\Users\jamio\shilladfs\product_semantic_search")

_ROOT = Path.cwd().resolve()
_STANDALONE = (_ROOT / "notebooks" / "image_split_standalone.py").resolve()
if not _STANDALONE.is_file():
    raise FileNotFoundError(f"독립 스크립트 없음: {_STANDALONE}")

_spec = importlib.util.spec_from_file_location("image_split_standalone", _STANDALONE)
_mod = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_mod)

set_runtime_config = _mod.set_runtime_config
get_local_image_list = _mod.get_local_image_list
get_image_list = _mod.get_image_list
process_images = _mod.process_images
save_outputs = _mod.save_outputs
log = _mod.log
print("loaded:", _STANDALONE)

In [ ]:
# 설정(전역) — 필요에 따라 수정
set_runtime_config(
    VALID_IMAGE_TILE_WIDTH=800,
    CONTENT_TILE_HEIGHT=900,
    SPLIT_OVERLAP_PX=30,
    MIN_LAST_TILE_HEIGHT_PX=200,
    OUTPUT_DIR=Path("app/data/image/output"),
    WHITE_THRESHOLD=245,
    USE_EDGE_BACKGROUND=True,
    SAVE_CONTENT_DEBUG_PREVIEW=True,
)

# 로컬 이미지 실행
_files, _images = get_local_image_list("app/data/image")
log(f"로드: files={len(_files)}, images={len(_images)}")

_merged, _tiles = process_images(_images)
_out = save_outputs(_merged)
log(f"저장 파일 수: {len(_out)}")
for p in _out:
    print(" -", p)

### S3 사용 시

- 환경 변수: `AWS_REGION` 또는 `AWS_DEFAULT_REGION`, 자격 증명(프로파일/역할 등)
- `s3://bucket/prefix/` 형태 또는 `prefix/` + `bucket=` 인자

In [ ]:
# 예시 (실제 버킷/프리픽스로 교체)
# import os
# os.environ["S3_BUCKET"] = "your-bucket"

# _b, _keys, _images = get_image_list("s3://your-bucket/path/to/detail/")
# _merged, _ = process_images(_images)
# _out = save_outputs(_merged)